In [ ]:
import os, sys, json, subprocess
import numpy as np
import pandas as pd

In [ ]:
BASE_DIR = os.environ.get("MELD_BASE_DIR", "/kaggle/working")
config = json.load(open(os.path.join(BASE_DIR, "config.json")))
RESULTS_DIR, EMOTIONS = config["paths"]["results_dir"], config["emotions"]
assert str(config.get("audio_feature_set", "")).startswith("wavlm") and str(config.get("video_backbone", "")).startswith("mtcnn"), "config.json describes the OLD features -- finish Steps 1-4 first."
print(f"BASE_DIR={BASE_DIR} | results={RESULTS_DIR}")

In [ ]:

VARIANTS = [   
    ("none",         {}),
    ("featmask0.1",  {"MELD_P9_FEAT_MASK": "0.1"}),
    ("uttmask0.15",  {"MELD_P9_UTT_MASK_P": "0.15"}),
    ("uttnoise0.15", {"MELD_P9_UTT_NOISE_P": "0.15"}),
    ("all_three",    {"MELD_P9_FEAT_MASK": "0.1", "MELD_P9_UTT_MASK_P": "0.15", "MELD_P9_UTT_NOISE_P": "0.15"}),
]
SEEDS = os.environ.get("MELD_SWEEP_SEEDS", "42,43,44")      
LAMBDA = os.environ.get("MELD_SWEEP_LAMBDA", "2.0")        
FOCUS = ["fear", "disgust", "sadness"]
PREFIX = "aug_"

def variant_env(tag, overrides, seeds, lam, base_env):
    env = dict(base_env)
    for k in ("MELD_P9_FEAT_MASK", "MELD_P9_UTT_MASK_P", "MELD_P9_UTT_NOISE_P"):
        env[k] = "0"                                       
    env.update({"MELD_P9_VERIFY": "0", "MELD_P9_SEEDS": seeds, "MELD_P9_LAMBDAS": lam, "MELD_P9_SWEEP_TAG": f"{PREFIX}{tag}"})
    env.update(overrides)
    return env

def summarize(results_dir, tags, emotions, focus=FOCUS, prefix=PREFIX):
    rows = []
    for tag in tags:
        d = os.path.join(results_dir, "sweeps", f"{prefix}{tag}")
        per_seed, idx = os.path.join(d, "phase9_val_per_seed.csv"), os.path.join(d, "phase9_run_index.json")
        if not (os.path.exists(per_seed) and os.path.exists(idx)):
            continue
        ps, run_index = pd.read_csv(per_seed, index_col=0), json.load(open(idx))
        summ = pd.read_csv(os.path.join(d, "phase9_val_summary.csv")).set_index("name")
        for name in ps.columns:
            cls = {e: [] for e in emotions}
            for r in run_index:
                if r["name"] != name:
                    continue
                log = [json.loads(l) for l in open(os.path.join(r["run_dir"], "log.jsonl"))]
                for e, f in zip(emotions, log[r["best_epoch"]]["val_f1_per_class"]):
                    cls[e].append(f)
            row = {"variant": tag, "model": "baseline" if name == "baseline" else "proposed", "n_seeds": int(ps[name].count()),
                   "val_macro_f1": float(ps[name].mean()), "val_macro_f1_std": float(ps[name].std(ddof=1)) if ps[name].count() > 1 else float("nan"),
                   "val_accuracy": float(summ.loc[name, "best_val_accuracy_mean"]),
                   "select_score": float(summ.loc[name, "best_select_score_mean"]) if "best_select_score_mean" in summ.columns else float("nan")}
            row.update({f"f1_{e}": float(np.mean(cls[e])) for e in focus if cls.get(e)})
            rows.append(row)
    df = pd.DataFrame(rows)
    if df.empty:
        return df, None, None
    def two_model(col):
        piv = df.pivot(index="variant", columns="model", values=col)
        piv["mean_of_two"] = piv.mean(axis=1)
        return piv.sort_values("mean_of_two", ascending=False)
    return df, two_model("val_macro_f1"), two_model("select_score")
print("Helpers defined.")

In [ ]:
RUNNER_SRC = """
import sys, nbformat
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError
src, dst, cwd = sys.argv[1:4]
nb = nbformat.read(src, as_version=4)
client = NotebookClient(nb, timeout=None, kernel_name="python3", resources={"metadata": {"path": cwd}})
ok = True
try:
    with client.setup_kernel():
        for i, cell in enumerate(nb.cells):
            if cell.cell_type != "code":
                continue
            client.execute_cell(cell, i)
            for o in cell.get("outputs", []):
                if o.get("output_type") == "stream":
                    t = o["text"]; print(t if len(t) < 3000 else t[:800] + "\\n...[truncated]...\\n" + t[-2000:], end="", flush=True)
except CellExecutionError as e:
    ok = False; print("CELL FAILED:", str(e)[-2500:], flush=True)
finally:
    nbformat.write(nb, dst)
sys.exit(0 if ok else 1)
"""
def find_phase9():
    for d in [os.environ.get("MELD_NOTEBOOK_DIR", ""), BASE_DIR, os.getcwd(), "/kaggle/working"]:
        if d and os.path.isfile(os.path.join(d, "Phase_09_Model_Training.ipynb")):
            return os.path.join(d, "Phase_09_Model_Training.ipynb")
    raise FileNotFoundError("Phase_09_Model_Training.ipynb not found (set MELD_NOTEBOOK_DIR). Use the STEP B version.")

nb9 = find_phase9()
assert "MELD_P9_UTT_MASK_P" in open(nb9).read(), "This Phase_09 notebook does not support the sweep (it lacks MELD_P9_SWEEP_TAG) -- use the STEP B version."
runner = os.path.join(BASE_DIR, "_run_notebook_sweep.py")
open(runner, "w").write(RUNNER_SRC)
os.makedirs(os.path.join(BASE_DIR, "executed_notebooks"), exist_ok=True)
for tag, ov in VARIANTS:
    done = os.path.join(RESULTS_DIR, "sweeps", f"{PREFIX}{tag}", "phase9_val_summary.csv")
    if os.path.exists(done):
        print(f"[{tag}] already complete -> skipped"); continue
    print(f"\n===== variant {tag}  ({ov or 'current loss'}) =====", flush=True)
    dst = os.path.join(BASE_DIR, "executed_notebooks", f"Phase_09_sweep_{tag}.ipynb")
    rc = subprocess.call([sys.executable, runner, nb9, dst, BASE_DIR], env=variant_env(tag, ov, SEEDS, LAMBDA, os.environ.copy()))
    if rc != 0:
        raise RuntimeError(f"variant {tag} failed (see {dst}). Fix the cause and re-run this notebook: finished variants/runs are skipped.")

In [ ]:
df, piv, piv_sel = summarize(RESULTS_DIR, [t for t, _ in VARIANTS], EMOTIONS)
pd.set_option("display.width", 220)
cols = ["variant", "model", "n_seeds", "val_macro_f1", "val_macro_f1_std", "val_accuracy"] + [f"f1_{e}" for e in FOCUS] + ["select_score"]
print(df[[c for c in cols if c in df.columns]].round(4).to_string(index=False))
print("\nValidation macro-F1 per variant (sorted by the mean of the two models):")
print(piv.round(4).to_string())
print("\nRobust selection score per variant (0.5 x clean + 0.5 x mean over degraded validation conditions; this is what picks the checkpoint):")
print(piv_sel.round(4).to_string())
df.to_csv(os.path.join(RESULTS_DIR, "augmentation_sweep_summary.csv"), index=False)
ref = piv.loc["none", "mean_of_two"] if "none" in piv.index else float("nan")
ref_s = piv_sel.loc["none", "mean_of_two"] if "none" in piv_sel.index else float("nan")
b1, b2 = piv.index[0], piv_sel.index[0]
print(f"\nBest by CLEAN validation macro-F1: {b1} ({piv.loc[b1, 'mean_of_two']:.4f}; legacy-only {ref:.4f}; difference {piv.loc[b1, 'mean_of_two'] - ref:+.4f})")
print(f"Best by ROBUST score:              {b2} ({piv_sel.loc[b2, 'mean_of_two']:.4f}; legacy-only {ref_s:.4f}; difference {piv_sel.loc[b2, 'mean_of_two'] - ref_s:+.4f})")
print("DECISION for you: which of the two rankings to follow. Per-seed std is about 0.01 -- differences below that are NOT distinguishable; then keep the legacy augmentation.")
print("Both models always share the same augmentation. Then run Phase 9 / Main with the chosen variant's MELD_P9_FEAT_MASK / MELD_P9_UTT_MASK_P / MELD_P9_UTT_NOISE_P.")